# Parametric HyperWaveNet Training

This notebook mounts Google Drive, copies one folder from `MyDrive` locally, rewrites audio references in your data config so they resolve against the copied Colab files, optionally installs the training repo from either GitHub or a zip in Drive while filtering macOS archive metadata, starts TensorBoard, and runs `nam-full-parametric` with live terminal-style output.

## What to put in Drive

Put one folder in Google Drive that contains:

- `input.wav`
- your reamped output wav files
- `data.json`
- `model.json`
- `learning.json`
- optionally, a repo zip such as `nam_parametric_repo.zip` (a Finder-created macOS zip is fine)

In the setup cell below, enter the folder path relative to `MyDrive/`, for example `parametric-training/amp_captures`.

In [ ]:
from pathlib import Path

DRIVE_FOLDER = "<folder path, do not include MyDrive>".strip("/")
if not DRIVE_FOLDER:
    raise ValueError("Set DRIVE_FOLDER to a folder under MyDrive, without a leading 'MyDrive/'.")
DATA_CONFIG_NAME = "data.json"
MODEL_CONFIG_NAME = "model.json"
LEARNING_CONFIG_NAME = "learning.json"

REPO_SOURCE = "github"  # "github" or "zip"
REPO_ZIP_NAME = "nam_parametric_repo.zip"  # used only when REPO_SOURCE == "zip"

REPO_URL = "https://github.com/phillipmself/neural-amp-modeler-parametric.git"
REPO_BRANCH = "feature/parametric-hypernetwork"

DRIVE_ROOT = Path("/content/drive/MyDrive")
DRIVE_DIR = DRIVE_ROOT / DRIVE_FOLDER
LOCAL_ROOT = Path("/content/nam_parametric")
LOCAL_DATA_ROOT = LOCAL_ROOT / "data"
RUNS_ROOT = LOCAL_ROOT / "runs"
REPO_DIR = LOCAL_ROOT / "repo"

LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

LOCAL_FOLDER_NAME = Path(DRIVE_FOLDER).name
LOCAL_DATA_DIR = LOCAL_DATA_ROOT / LOCAL_FOLDER_NAME
DRIVE_RUNS_ROOT = DRIVE_DIR / "runs"
SYNC_INTERVAL_SECONDS = 60

print(f"Drive folder: MyDrive/{DRIVE_FOLDER}")
print(f"Repo source: {REPO_SOURCE}")
if REPO_SOURCE == "zip":
    print(f"Repo zip: {REPO_ZIP_NAME}")
print(f"Local data dir: {LOCAL_DATA_DIR}")
print(f"Runs root: {RUNS_ROOT}")
print(f"Drive runs root: {DRIVE_RUNS_ROOT}")

In [ ]:
import shutil
import subprocess
import sys
import zipfile
from pathlib import PurePosixPath

def _is_archive_metadata_name(name: str) -> bool:
    return name in {"__MACOSX", ".DS_Store"} or name.startswith(".") or name.startswith("._")

def _is_archive_metadata_path(path: Path | PurePosixPath) -> bool:
    return any(_is_archive_metadata_name(part) for part in path.parts if part not in {"", "."})

def _looks_like_repo_root(path: Path) -> bool:
    return (path / "pyproject.toml").exists() and (path / "nam" / "train" / "parametric.py").exists()

def _resolve_repo_source_dir(extract_root: Path) -> Path:
    if _looks_like_repo_root(extract_root):
        return extract_root
    candidates = sorted(
        {
            path.parent
            for path in extract_root.rglob("pyproject.toml")
            if not _is_archive_metadata_path(path.relative_to(extract_root))
        },
        key=lambda path: (len(path.relative_to(extract_root).parts), str(path)),
    )
    for candidate in candidates:
        if _looks_like_repo_root(candidate):
            return candidate
    raise FileNotFoundError(
        "Couldn't find the repo root in the extracted archive. Expected pyproject.toml and "
        f"nam/train/parametric.py under {extract_root}"
    )

def _resolve_repo_zip_path() -> Path:
    if not REPO_ZIP_NAME:
        raise ValueError("Set REPO_ZIP_NAME when REPO_SOURCE is 'zip'")
    direct_path = DRIVE_DIR / REPO_ZIP_NAME
    if direct_path.exists():
        return direct_path
    matches = sorted(
        path
        for path in DRIVE_DIR.rglob(REPO_ZIP_NAME)
        if not _is_archive_metadata_path(path.relative_to(DRIVE_DIR))
    )
    if len(matches) == 1:
        return matches[0]
    if not matches:
        raise FileNotFoundError(f"Repo zip not found under: {DRIVE_DIR}")
    raise FileExistsError(f"Found multiple repo zips named {REPO_ZIP_NAME!r} under {DRIVE_DIR}")

def _extract_repo_zip(repo_zip_path: Path, extract_root: Path) -> None:
    if extract_root.exists():
        shutil.rmtree(extract_root)
    extract_root.mkdir(parents=True, exist_ok=True)
    kept_members = []
    with zipfile.ZipFile(repo_zip_path) as archive:
        for info in archive.infolist():
            relative_path = PurePosixPath(info.filename)
            if _is_archive_metadata_path(relative_path):
                continue
            if any(part == ".." for part in relative_path.parts):
                raise ValueError(f"Unsafe path inside repo zip: {info.filename}")
            archive.extract(info, extract_root)
            kept_members.append(info.filename)
    if not kept_members:
        raise FileNotFoundError(f"No usable files found in repo zip: {repo_zip_path}")

extract_root = LOCAL_ROOT / "repo_extract"
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
if extract_root.exists():
    shutil.rmtree(extract_root)

if REPO_SOURCE == "github":
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )
elif REPO_SOURCE == "zip":
    from google.colab import drive

    drive.mount("/content/drive", force_remount=True)
    repo_zip_path = _resolve_repo_zip_path()
    if not zipfile.is_zipfile(repo_zip_path):
        raise ValueError(f"Repo archive must be a zip file: {repo_zip_path}")
    _extract_repo_zip(repo_zip_path, extract_root)
    repo_source_dir = _resolve_repo_source_dir(extract_root)
    shutil.move(str(repo_source_dir), str(REPO_DIR))
else:
    raise ValueError("REPO_SOURCE must be 'github' or 'zip'")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)

import torch
import pytorch_lightning as pl
import tensorboard

print(f"Resolved repo directory: {REPO_DIR}")
print(f"Torch: {torch.__version__}")
print(f"PyTorch Lightning: {pl.__version__}")
print(f"TensorBoard: {tensorboard.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Installed repo from: {REPO_SOURCE}")

In [ ]:
import json
import shutil
from pathlib import PurePosixPath, PureWindowsPath
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

if not DRIVE_DIR.exists():
    raise FileNotFoundError(f"Drive folder not found: {DRIVE_DIR}")

def _ignore_drive_copy(dirpath: str, names: list[str]) -> set[str]:
    current_dir = Path(dirpath)
    return {
        name
        for name in names
        if _is_archive_metadata_name(name)
        or (current_dir == DRIVE_DIR and name == "runs")
        or (REPO_SOURCE == "zip" and name == REPO_ZIP_NAME)
    }

if LOCAL_DATA_DIR.exists():
    shutil.rmtree(LOCAL_DATA_DIR)
shutil.copytree(DRIVE_DIR, LOCAL_DATA_DIR, ignore=_ignore_drive_copy)
DRIVE_RUNS_ROOT.mkdir(parents=True, exist_ok=True)

data_config_path = LOCAL_DATA_DIR / DATA_CONFIG_NAME
model_config_path = LOCAL_DATA_DIR / MODEL_CONFIG_NAME
learning_config_path = LOCAL_DATA_DIR / LEARNING_CONFIG_NAME

for path in (data_config_path, model_config_path, learning_config_path):
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

def _load_json_config(path: Path) -> dict:
    try:
        return json.loads(path.read_text())
    except json.JSONDecodeError as exc:
        raise ValueError(
            f"Invalid JSON in {path.name} at line {exc.lineno}, column {exc.colno}: {exc.msg}"
        ) from exc

data_config = _load_json_config(data_config_path)
model_config = _load_json_config(model_config_path)
learning_config = _load_json_config(learning_config_path)

def _candidate_local_audio_paths(raw_path: str) -> list[Path]:
    candidates = []
    posix_path = PurePosixPath(raw_path)
    windows_path = PureWindowsPath(raw_path)

    def _append(candidate: Path) -> None:
        if candidate not in candidates:
            candidates.append(candidate)

    if not windows_path.is_absolute() and not posix_path.is_absolute():
        relative_parts = [part for part in posix_path.parts if part not in {"", "."}]
        if relative_parts:
            _append(LOCAL_DATA_DIR / Path(*relative_parts))

    basename = windows_path.name or posix_path.name
    if basename:
        _append(LOCAL_DATA_DIR / basename)

    for pure_path in (windows_path, posix_path):
        if pure_path.is_absolute():
            anchor = pure_path.anchor
            relative_parts = [part for part in pure_path.parts if part not in {"", ".", anchor}]
            if relative_parts:
                _append(LOCAL_DATA_DIR / Path(*relative_parts))

    return candidates

def _rewrite_audio_path(raw_path: str) -> str:
    candidates = _candidate_local_audio_paths(raw_path)
    for candidate in candidates:
        if candidate.exists():
            return str(candidate)
    attempted = "\n".join(f"  - {candidate}" for candidate in candidates)
    raise FileNotFoundError(
        f"Couldn't map audio path {raw_path!r} into the copied Drive folder {LOCAL_DATA_DIR}. Tried:\n{attempted}"
    )

resolved_audio_paths: set[str] = set()

def _rewrite_path_fields(mapping: dict) -> None:
    for key in ("x_path", "y_path"):
        raw_path = mapping.get(key)
        if raw_path is None:
            continue
        resolved_path = _rewrite_audio_path(raw_path)
        mapping[key] = resolved_path
        resolved_audio_paths.add(resolved_path)

common = data_config.get("common", {})
if isinstance(common, dict):
    _rewrite_path_fields(common)

def _rewrite_section(section_name: str) -> None:
    section = data_config.get(section_name)
    if isinstance(section, dict):
        _rewrite_path_fields(section)
        return
    if isinstance(section, list):
        for item in section:
            if isinstance(item, dict):
                _rewrite_path_fields(item)

for section_name in ("train", "validation"):
    _rewrite_section(section_name)

with open(data_config_path, "w") as fp:
    json.dump(data_config, fp, indent=4)

def _sync_tree(source_root: Path, dest_root: Path) -> None:
    dest_root.mkdir(parents=True, exist_ok=True)
    for path in source_root.rglob("*"):
        relative = path.relative_to(source_root)
        target = dest_root / relative
        if path.is_dir():
            target.mkdir(parents=True, exist_ok=True)
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists():
            src_stat = path.stat()
            dst_stat = target.stat()
            if (
                src_stat.st_size == dst_stat.st_size
                and src_stat.st_mtime_ns <= dst_stat.st_mtime_ns
            ):
                continue
        shutil.copy2(path, target)

print("Copied folder locally and rewrote config audio paths.")
print(f"Resolved {len(resolved_audio_paths)} audio file paths.")
print(f"Data config: {data_config_path}")
print(f"Model config: {model_config_path}")
print(f"Learning config: {learning_config_path}")
print("Validated JSON syntax for data/model/learning configs.")
print(f"Run artifacts will sync to: {DRIVE_RUNS_ROOT}")

In [ ]:
%load_ext tensorboard
%tensorboard --logdir $RUNS_ROOT

In [ ]:
import errno
import os
import pty
import select
import subprocess
import sys
import textwrap
import time

before_runs = {path.name for path in RUNS_ROOT.iterdir() if path.is_dir()}

def _sync_runs_to_drive() -> None:
    _sync_tree(RUNS_ROOT, DRIVE_RUNS_ROOT)

_sync_runs_to_drive()
TRAIN_SNIPPET = textwrap.dedent(
    '''
    import importlib
    import inspect
    import json
    import sys
    from pathlib import Path

    from nam.util import timestamp

    module = importlib.import_module("nam.train.parametric")
    entrypoint = None
    for name in ("main", "train", "run"):
        candidate = getattr(module, name, None)
        if callable(candidate):
            entrypoint = candidate
            entrypoint_name = name
            break
    if entrypoint is None:
        available = sorted(name for name in dir(module) if not name.startswith("_"))
        raise ImportError(
            "Couldn't find a callable parametric training entrypoint in "
            f"nam.train.parametric. Tried main/train/run; available symbols: {available}"
        )

    def load_json(path_str):
        path = Path(path_str)
        try:
            return json.loads(path.read_text())
        except json.JSONDecodeError as exc:
            raise ValueError(
                f\"Invalid JSON in {path.name} at line {exc.lineno}, column {exc.colno}: {exc.msg}\"
            ) from exc

    data_config = load_json(sys.argv[1])
    model_config = load_json(sys.argv[2])
    learning_config = load_json(sys.argv[3])
    outdir = Path(sys.argv[4], timestamp())
    outdir.mkdir(parents=True, exist_ok=False)
    flags = set(sys.argv[5:])
    kwargs = {
        "data_config": data_config,
        "model_config": model_config,
        "learning_config": learning_config,
        "outdir": outdir,
        "no_show": "--no-show" in flags,
        "make_plots": "--no-plots" not in flags,
    }
    signature = inspect.signature(entrypoint)
    accepted_kwargs = {
        name: value for name, value in kwargs.items() if name in signature.parameters
    }
    missing_required = [
        name
        for name, parameter in signature.parameters.items()
        if parameter.default is inspect._empty
        and parameter.kind
        in (
            inspect.Parameter.POSITIONAL_ONLY,
            inspect.Parameter.POSITIONAL_OR_KEYWORD,
            inspect.Parameter.KEYWORD_ONLY,
        )
        and name not in accepted_kwargs
    ]
    if missing_required:
        raise TypeError(
            f"Resolved nam.train.parametric.{entrypoint_name}, but its signature "
            f"{signature} is incompatible with the notebook. Missing parameters: {missing_required}"
        )
    entrypoint(**accepted_kwargs)
    '''
)

command = [
    sys.executable,
    "-c",
    TRAIN_SNIPPET,
    str(data_config_path),
    str(model_config_path),
    str(learning_config_path),
    str(RUNS_ROOT),
    "--no-show",
]

master_fd, slave_fd = pty.openpty()
process = subprocess.Popen(
    command,
    cwd=REPO_DIR,
    stdin=subprocess.DEVNULL,
    stdout=slave_fd,
    stderr=slave_fd,
    env={**os.environ, "PYTHONUNBUFFERED": "1"},
    text=False,
)
os.close(slave_fd)

last_sync = time.monotonic()

try:
    while True:
        ready, _, _ = select.select([master_fd], [], [], 0.1)
        if master_fd in ready:
            try:
                chunk = os.read(master_fd, 4096)
            except OSError as exc:
                if exc.errno == errno.EIO:
                    break
                raise
            if chunk:
                print(chunk.decode("utf-8", errors="replace"), end="")
            else:
                break
        now = time.monotonic()
        if now - last_sync >= SYNC_INTERVAL_SECONDS:
            _sync_runs_to_drive()
            last_sync = now
        if process.poll() is not None and master_fd not in ready:
            break
finally:
    os.close(master_fd)
    _sync_runs_to_drive()

return_code = process.wait()
_sync_runs_to_drive()
if return_code != 0:
    raise RuntimeError(f"Training failed with exit code {return_code}")

after_runs = sorted(path for path in RUNS_ROOT.iterdir() if path.is_dir())
new_runs = [path for path in after_runs if path.name not in before_runs]
LATEST_RUN_DIR = new_runs[-1] if new_runs else max(after_runs, key=lambda path: path.stat().st_mtime)
LATEST_DRIVE_RUN_DIR = DRIVE_RUNS_ROOT / LATEST_RUN_DIR.name
print()
print(f"Training complete. Latest local run directory: {LATEST_RUN_DIR}")
print(f"Latest Drive run directory: {LATEST_DRIVE_RUN_DIR}")

In [ ]:
if "LATEST_RUN_DIR" not in globals():
    raise RuntimeError("Run the training cell first.")

for path in sorted(LATEST_RUN_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(LATEST_RUN_DIR))